# CyberSentinel — GRPO on RunPod (RTX A6000 48GB)

Interactive companion to **`train_runpod.py`**. Every cell calls into that module, so
the notebook and the headless script cannot drift apart — edit the logic there, drive
it from here.

| | |
|---|---|
| Policy | `unsloth/Qwen3-4B-Instruct-2507` (Apache-2.0, non-thinking) |
| Dataset | `tumeteor/Security-TTP-Mapping` (train 14,900 rows) |
| GPU | 1x RTX A6000, 48GB, Ampere SM 8.6, CUDA 12.4 |
| Precision | **bfloat16** (native on Ampere) |
| Storage | `/workspace/...` (RunPod persistent volume) |

**For a long run, prefer the script inside tmux** — a JupyterLab tab that loses its
websocket can take the kernel with it:

```bash
tmux new -s cti
python train_runpod.py 2>&1 | tee /workspace/train.log
# detach: Ctrl-b then d
```

Use this notebook for inspecting the data, sanity-checking rewards, and short runs.

**Run the cells in order** — the environment variables in step 2 must be set before
Unsloth is imported in step 4.

## 1. Setup

In [ ]:
# train_runpod.py must sit next to this notebook (both live in the repo root).
import os
import sys

sys.path.insert(0, os.getcwd())

try:
    import train_runpod as T
except ModuleNotFoundError as exc:
    raise SystemExit(
        f"{exc}\n\nCould not import train_runpod.py from {os.getcwd()}.\n"
        "Upload it next to this notebook, e.g.:\n"
        "  cd /workspace && git clone <your repo> && cd CyberSentinel"
    )

print("module   :", T.__file__)
print("workspace:", T.WORKSPACE)
for name, path in [("outputs", T.OUTPUT_DIR), ("adapters", T.ADAPTER_DIR),
                   ("merged", T.MERGED_DIR), ("gguf", T.GGUF_DIR)]:
    print(f"  {name:9s}: {path}")

## 2. Configuration

`parse_args([])` gives the documented defaults; pass flags to override exactly as you
would on the command line. All of these are also settable from the shell — see
`python train_runpod.py --help`.

In [ ]:
# Defaults: G=4, grad_accum=4, 2000 steps, bf16, gpu_memory_utilization=0.85.
# Examples:
#   args = T.parse_args(["--full-epoch"])              # ~3725 steps (one epoch)
#   args = T.parse_args(["--num-generations", "8"])    # 48GB allows a larger group
#   args = T.parse_args(["--allow-synthetic", "--max-steps", "5"])  # plumbing test
args = T.parse_args([])

# Sets UNSLOTH_VLLM_STANDBY / CUDA_VISIBLE_DEVICES and creates /workspace dirs.
# MUST run before Unsloth is imported (step 4).
#
# Note what is NOT set: UNSLOTH_VLLM_NO_FLASHINFER. That was a Kaggle T4 workaround
# for a vLLM JIT link failure ("cannot find -lcuda"); on Ampere + CUDA 12.4 the
# native FlashInfer sampling kernels build fine and are faster.
T.configure_environment(args)

T.report_gpu()
precision = T.resolve_precision(args.precision)   # -> "bf16" on Ampere
print("precision:", precision)

## 3. Reward smoke test

Pure Python, no GPU, milliseconds. A broken reward function is invisible during
training — you just watch a flat line for hours — so verify the scoring first.

In [ ]:
T.run_reward_smoke_test()

# What the three rewards do, on one example each:
good = [{"role": "assistant",
         "content": "<reasoning>encoded powershell child process</reasoning>\n"
                    "<answer>T1059.001</answer>"}]
gt = [["T1059.001"]]
print("  format     :", T.format_reward_func(None, [good]))
print("  correctness:", T.correctness_reward_func(None, [good], ground_truth=gt))
print("  soft_format:", T.soft_format_reward_func(None, [good]))

# Exact match matters: the parent technique does NOT satisfy a sub-technique label.
parent = [{"role": "assistant",
           "content": "<reasoning>powershell</reasoning>\n<answer>T1059</answer>"}]
print("  T1059 vs T1059.001 ->", T.correctness_reward_func(None, [parent], ground_truth=gt))

## 4. Load the policy

4-bit base + LoRA rank 16, generation routed through vLLM. `max_seq_length=1024` is
load-bearing: Qwen3-4B advertises a 262,144-token context and vLLM sizes its KV cache
from `max_model_len`, so leaving it uncapped tries to allocate a KV cache for 262k
tokens and OOMs before step 1.

In [ ]:
model, tokenizer = T.load_policy(args, precision)

## 5. Dataset

`text1` (CTI snippet) + `labels` (stringified list like `"['T1547.001']"`).

Two deliberate behaviours: a schema or parse error **raises** rather than silently
falling back to 8 synthetic rows (a run that learns nothing looks identical to a good
one until the metrics come back), and prompts over `max_prompt_length` are dropped
because TRL truncates from the *left*, which would eat the system prompt the format
reward grades against.

In [ ]:
dataset = T.drop_overlong_prompts(
    T.build_dataset(args.dataset, args.split, args.allow_synthetic),
    tokenizer,
    args.max_prompt_length,
)

assert len(dataset) >= args.min_rows, (
    f"Only {len(dataset)} rows survived filtering — expected ~14,900. "
    "Stop and investigate before burning GPU-hours."
)

print(f"\nTRAIN ROWS: {len(dataset)}")
print("\nexample prompt:\n")
print(dataset[0]["prompt"][1]["content"][:400])
print("\nground_truth:", dataset[0]["ground_truth"])

## 6. Trainer

`temperature=0.9` is the knob that matters most. GRPO's advantage is
`(r_i - mean(r)) / std(r)` computed **within** each group of G completions. Sample too
greedily and all G come out near-identical, `std(r) ~ 0`, every advantage collapses to
~0, and the step contributes no gradient — the run looks stable while learning nothing.

In [ ]:
from trl import GRPOTrainer

training_args = T.build_training_args(args, precision, len(dataset))

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=T.REWARD_FUNCS,   # format (0/1), correctness (-1/+2), soft (0..0.5)
    args=training_args,
    train_dataset=dataset,
)

## 7. Train

**Watch `reward_std` first.** If it sits near 0 for the first few dozen steps the group
has collapsed and nothing is being learned — raise `--temperature` and restart rather
than letting it burn hours.

Then `rewards/format_reward_func` should climb toward 1.0 (the easy signal), and
`rewards/correctness_reward_func` lifting off its -1.0 floor is the real evidence that
policy search is finding correct technique IDs. A `kl` spike means raise `--beta`.

Checkpoints land every 100 steps (keeping 3), so a dropped connection is recoverable:
re-run with `resume_from_checkpoint=True`.

In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint=args.resume)
print(trainer_stats)

## 8. Save artifacts

Three outputs, three consumers:

| Path | What | Used by |
|---|---|---|
| `/workspace/grpo_cti_adapters` | PEFT LoRA + tokenizer | `evaluate_cti_agent.py` (needs CUDA) |
| `/workspace/grpo_cti_merged` | merged 16-bit | input for GGUF conversion |
| `/workspace/grpo_cti_gguf` | Q4_K_M ~2.5GB | local serving on any laptop |

Keep the merged directory even after a successful GGUF export — it lets you redo the
conversion without retraining.

In [ ]:
T.save_artifacts(model, tokenizer)

## 9. Export GGUF

Tries Unsloth's one-liner, then falls back to cloning llama.cpp and driving
`convert_hf_to_gguf.py` + a CMake build of `llama-quantize` (there is a known upstream
issue where `save_pretrained_gguf` rejects `quantization_method`). Only the quantiser
target is built, with `GGML_CUDA=OFF`, because quantisation is a CPU job.

Either way the result is `/workspace/grpo_cti_gguf/cybersentinel-cti-q4_k_m.gguf`.

In [ ]:
gguf_path = T.export_gguf(
    model, tokenizer,
    quant=args.quant,               # q4_k_m (~2.5GB); q5_k_m (~2.9GB) if quality drops
    merged_dir=args.merged_dir,
    gguf_dir=args.gguf_dir,
)
T.report_gguf(args.gguf_dir)
print(T.NEXT_STEPS.format(gguf=gguf_path, name=os.path.basename(gguf_path)))

## 10. Evaluate (optional)

Format adherence / accuracy / reasoning length against the probe set. Needs CUDA
(bitsandbytes 4-bit), so run it here on the pod rather than on a laptop:

```bash
python evaluate_cti_agent.py \
    --adapter_path /workspace/grpo_cti_adapters \
    --base_model Qwen/Qwen3-4B-Instruct-2507
```

Worth doing **before** you trust the quantised file: compare these numbers against the
GGUF's, since Q4_K_M trades some accuracy for size. If format adherence drops, re-export
at `--quant q5_k_m`.